# Why the tick-array book was not faster

The performance ladder in `unito26.lob.orderbook` ends in `TickArrayBook`, which fuses the
sizes and the occupancy index into one tick-indexed array. It is meant to be the fastest
rung. Measured as the session notebook measured it, it was the *slowest*.

Nothing was wrong with the book. Three things were wrong with the measurement, and each is
worth more than the speedup it hides:

1. the band was sized from a message stream that carries a **sentinel price of zero**;
2. the book being folded held about a dozen occupied levels a side, which is the regime
   where an index has nothing to win;
3. the fold asked the book for the same thing four times per message, so most of what was
   being timed was not the ladder at all.

This notebook is the diagnosis, in that order, and then the measurements that justify each
change made in response — including three ideas that were measured and **rejected**.

In [ ]:
import time
from itertools import accumulate

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from unito26.lob import config, frames, visualization
from unito26.lob.benchmark import REFERENCE_PRICE
from unito26.lob.messages import (
    BUY, SELL, MARKET_BUY_PRICE, MARKET_SELL_PRICE,
    GridDepth, ReportedDepth, SweepSize, is_market_price,
)
from unito26.lob.orderbook import AXIS_B_VARIANTS, AggregateBook, TickArrayBook
from unito26.lob.session import MarketSession
from unito26.lob.statistics import SessionStatistics
from unito26.lob.simulate import OrderFlowSimulator

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"
pio.templates["unito26"] = go.layout.Template(layout=dict(
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(color=INK, size=12),
    xaxis=dict(gridcolor=GRID, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)),
    yaxis=dict(gridcolor=GRID, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)),
))
pio.templates.default = "unito26"

DEPTH = ReportedDepth(10)
LEVELS = tuple(GridDepth(n) for n in (1, 2, 3, 5, 10))
SWEEPS = tuple(SweepSize(q) for q in (100, 400, 1600))
WINDOWS = (1, 10, 60)          # seconds
SPEC = SessionStatistics(LEVELS, SWEEPS, WINDOWS)
PRICE_UNIT = 100


def best_of(call, repeat=3):
    """Best of `repeat`: the left tail is the machine doing only our work."""
    fastest = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        call()
        fastest = min(fastest, time.perf_counter() - start)
    return fastest


def simulate(mark_params, horizon, seed):
    """A warmed book and the stream that follows it, as the session notebook builds them."""
    simulator = OrderFlowSimulator(
        config.example_order_flow_params(), mark_params, REFERENCE_PRICE, rng=seed
    )
    book = AggregateBook()
    simulator.warm_up(book, horizon=60.0, journal=None)
    opening = book.copy()
    messages = []
    for message in simulator.stream(book, horizon=horizon, journal=None):
        book.apply(message, record=False)
        messages.append(message)
    return opening, messages, book


SHALLOW = simulate(config.example_mark_params(), 300.0, 11)
DEEP = simulate(config.deep_mark_params(), 300.0, 11)

REGIMES = {"shallow": SHALLOW, "deep": DEEP}
pd.DataFrame({
    name: {
        "messages": len(messages),
        "occupied levels a side": (len(final.levels_map(BUY)) + len(final.levels_map(SELL))) // 2,
        "price range (ticks)": max(p for p in (m.price for m in messages) if not is_market_price(p))
                             - min(p for p in (m.price for m in messages) if not is_market_price(p)),
    }
    for name, (_, messages, final) in REGIMES.items()
})

Two regimes, from parametrizations the package already ships. They differ only in
`DepthDecay`: `example_mark_params` clusters orders at the touch, `deep_mark_params`
spreads them. The second holds about ten times as many occupied levels a side, and that
number is the whole subject of section 3.

## 1. What the ladder times, and what the fold pays for

`AggregateBook.apply` is the matching: find the best price on the opposite side, consume it,
rest the remainder. That is what the ladder was built to make faster. Time it alone, then
time it inside a fold that also records the book after every message.

In [ ]:
def band(opening, messages):
    """Every price the run touches.  `for_prices` drops the sentinels itself."""
    return ([m.price for m in messages]
            + list(opening.levels_map(BUY)) + list(opening.levels_map(SELL)))


def seeded(book_cls, opening, messages):
    book = book_cls.for_prices(band(opening, messages))
    for direction in (BUY, SELL):
        for price, resting in opening.levels_map(direction).items():
            book.set_size(direction, price, resting)
    return book


def bare_fold(book_cls, opening, messages):
    book = seeded(book_cls, opening, messages)
    for message in messages:
        book.apply(message, record=False)


rows = {}
for name, (opening, messages, _) in REGIMES.items():
    rows[name] = {
        "apply alone": best_of(lambda: bare_fold(AggregateBook, opening, messages)),
        "+ the LOBSTER rows": best_of(lambda: MarketSession.from_occupied_levels(
            seeded(AggregateBook, opening, messages), messages, DEPTH, SPEC, PRICE_UNIT, False)),
        "+ the statistics": best_of(lambda: MarketSession.from_occupied_levels(
            seeded(AggregateBook, opening, messages), messages, DEPTH, SPEC, PRICE_UNIT, True)),
    }
pd.DataFrame(rows).round(4)

Matching is a rounding error. Everything else is recording — asking the book what it looks
like now, and turning that into a row and a set of statistics. A ladder that makes
`best_price` faster is optimising about two per cent of the fold.

That does not make the ladder pointless; it relocates it. Recording *also* goes through the
book, by way of `occupied_levels`, and that is where the index earns its keep. But it means
the fold has to be measured, not `apply`.

## 2. The sentinel that widened the band sixty-fold

A market order carries a price specification guaranteeing execution, not a position on the
grid. There are two of them and **they fail differently**, which is how a filter written for
one comes to let the other through.

In [ ]:
print(f"MARKET_SELL_PRICE = {MARKET_SELL_PRICE}")
print(f"MARKET_BUY_PRICE  = {MARKET_BUY_PRICE}")

opening, messages, _ = SHALLOW
raw = [m.price for m in messages]
loose = [p for p in raw if p < 10 ** 9]        # the filter this notebook used to carry
tight = [p for p in raw if not is_market_price(p)]
print(f"\nprices, unfiltered      {min(raw)} .. {max(raw)}")
print(f"prices, p < 10**9       {min(loose)} .. {max(loose)}")
print(f"prices, is_market_price {min(tight)} .. {max(tight)}")

`p < 10 ** 9` removes `MARKET_BUY_PRICE`, because `sys.maxsize` is conspicuous. It does not
remove `MARKET_SELL_PRICE`, because that is `0` — and, as `AggregateBook.resting_price` says
in its own docstring, 0 is indistinguishable from a real price.

So the band ran from 0 to the top of the market instead of across the market, and both
bitmap-backed rungs carried a ten-thousand-bit integer that every write had to rewrite.
`AggregateBook.for_prices` now drops both sentinels itself, so the failure is no longer
available to a caller. The cost it used to carry is still measurable, by building the two
bands by hand.

In [ ]:
def at_band(width, opening, messages):
    """The same book and the same stream, in a band of a chosen width."""
    prices = [p for p in (m.price for m in messages) if not is_market_price(p)]
    low, high = min(prices), max(prices)
    slack = max(0, width - (high - low)) // 2
    book = TickArrayBook(origin=low - slack - 64, width=(high - low) + 2 * slack + 128)
    for direction in (BUY, SELL):
        for price, resting in opening.levels_map(direction).items():
            book.set_size(direction, price, resting)
    return book


rows = []
for name, (opening, messages, _) in REGIMES.items():
    timings = {}
    for label, width in (("across the market", 0), ("from zero", 10_000)):
        timings[f"{label}: ticks"] = at_band(width, opening, messages).width
        timings[f"{label}: seconds"] = round(best_of(
            lambda w=width: MarketSession.from_occupied_levels(
                at_band(w, opening, messages), messages, DEPTH, SPEC, PRICE_UNIT, False)
        ), 4)
    rows.append(pd.Series(timings, name=name))

table = pd.DataFrame(rows)
table["cost of the sentinel"] = (
    table["from zero: seconds"] / table["across the market: seconds"]
).round(2)
table

The band is not free even when nothing rests in it: it is the width of the integer the
occupancy lives in, and Python integers are immutable, so every `set_size` allocates and
copies the whole thing.

The ratio above is around 1.1, where before section 6 was written it was nearer 1.75. That is
the mirrored ask side doing its work: the operation a wide band used to punish -- finding the
*lowest* set bit -- is no longer performed at all. The band still costs something, and it
costs nothing to stop paying it.

## 3. The regime where no index can win

With the shallow parametrization the book holds about a dozen occupied levels a side.
`max()` over a dozen dictionary keys is a single loop in C. There is nothing there for a
bitmap to beat — and the session notebook was measuring the ladder there.

In [ ]:
rows = []
for name, (opening, messages, final) in REGIMES.items():
    occupied = (len(final.levels_map(BUY)) + len(final.levels_map(SELL))) // 2
    timings = {"occupied levels a side": occupied}
    for book_cls in AXIS_B_VARIANTS:
        timings[book_cls.__name__] = round(best_of(
            lambda c=book_cls: MarketSession.from_occupied_levels(
                seeded(c, opening, messages), messages, DEPTH, SPEC, PRICE_UNIT, True)
        ), 3)
    rows.append(pd.Series(timings, name=name))

pd.DataFrame(rows)

The dict-backed rungs roughly double from one regime to the other; the tick array barely
moves. That is the ladder's claim, stated properly: **the index is worth what the scan
costs, and the scan costs what the book is deep.**

A shallow book is not an unrealistic book — a liquid large-cap with a one-tick spread looks
like the first row. It is simply the case where the answer is "use the dictionary".

## 4. What the fold was actually doing

Before the changes below, folding one message asked the book to walk its occupancy four
times: once per side for the statistics, and once per side again inside `write_lobster_row`.
On top of that `queue_imbalance` was called once per entry of `imbalance_levels`, and each
call re-derived both best prices and re-summed a window that nests inside the next.

In [ ]:
import cProfile, io, pstats

opening, messages, _ = DEEP
profiler = cProfile.Profile()
profiler.enable()
MarketSession.from_occupied_levels(
    seeded(TickArrayBook, opening, messages), messages, DEPTH, SPEC, PRICE_UNIT, True
)
profiler.disable()
report = io.StringIO()
pstats.Stats(profiler, stream=report).sort_stats("tottime").print_stats(8)
print("\n".join(report.getvalue().splitlines()[4:16]))
print(f"\n{len(messages)} messages, {len(LEVELS)} imbalance levels")

Three changes follow from that count, and all three are ways of *not asking twice*.

**One walk a side.** `SideStatistics` already carries `levels`, and `levels` **is**
`occupied_levels(direction, reported_depth)` — the same call. So the LOBSTER row is written
from what the statistics already walked for, rather than walked for again.

**The imbalance in one pass.** $I^1, I^2, \dots$ nest: the first $n$ grid positions contain
the first $n-1$. So the deepest walk carries every shallower answer in its running total,
and `queue_imbalance_profile` returns the whole tuple from one walk of each side. On the
array-backed rung the walk is a slice of the size array and the running totals are
`itertools.accumulate`, so the accumulation never enters Python.

**The gaps from the prices already in hand.** Two consecutive occupied prices bound exactly
one maximal run of empty positions, of length $|p_{i+1} - p_i| - 1$. So the whole gap
structure is a by-product of the walk that produced `levels` — no rung has to enumerate grid
positions, mask an integer, or count bits.

That last one has a casualty, and it is worth being straight about it.

In [ ]:
# `count_binary_gaps` and `measure_largest_binary_gap` still answer the same question, and
# the individual methods still go through them.  They are simply no longer the fast way.
book = TickArrayBook.from_levels({100: 5, 97: 6, 93: 7}, {110: 8, 111: 9})
depth = ReportedDepth(3)
side = book.side_statistics(BUY, depth)
print("from the levels list :", side.gap_count, side.largest_gap)
print("from the bitmap      :",
      book.gap_count(BUY, depth),
      book.largest_gap_size_between_non_empty_levels(BUY, depth))

def by_bitmap(book, depth, repeat=4000):
    for _ in range(repeat):
        for direction in (BUY, SELL):
            book.gap_count(direction, depth)
            book.largest_gap_size_between_non_empty_levels(direction, depth)

def by_levels(book, depth, repeat=4000):
    for _ in range(repeat):
        for direction in (BUY, SELL):
            book.side_statistics(direction, depth)

opening, messages, final = DEEP
deep_book = seeded(TickArrayBook, opening, messages)
for message in messages[:4000]:
    deep_book.apply(message, record=False)
print(f"\ncounting gaps off the occupancy integer : {best_of(lambda: by_bitmap(deep_book, DEPTH)):.3f}s")
print(f"reading them off the levels already walked: {best_of(lambda: by_levels(deep_book, DEPTH)):.3f}s")

The bit tricks answer a question the walk has already answered. That is a better lesson than
the one the section used to carry: an optimisation that beats the obvious code, and then a
simpler restatement of the problem that beats them both.

`documentation/integers-in-binary.md` still derives the identities, and
`unito26.lob.binary_gaps` still has its own tests. What changed is which one the book calls.

## 5. Where the gaps are, not just how many

`GapCount` and `LargestGap` say a side has holes and how long the longest is. They do not say
*where*. Three more statistics, all falling out of the same single pass:

| column | meaning |
| --- | --- |
| `{Side}FirstGapDistance` | ticks from the touch to the shallowest empty position of the nearest gap |
| `{Side}FirstGapSize` | levels in that nearest gap |
| `{Side}LargestGapDistance` | the same distance for the largest gap, nearest of them if two tie |

A distance is `None` on a book and NaN in a frame when the side has no gap: there is no
position to name, and zero would name the touch, which is always occupied.

In [ ]:
opening, messages, _ = DEEP
session = MarketSession.from_occupied_levels(
    seeded(TickArrayBook, opening, messages), messages, DEPTH, SPEC, PRICE_UNIT, True
)
placement = [f"{side}{name}" for side in ("Bid", "Ask")
             for name in ("FirstGapDistance", "FirstGapSize", "LargestGapDistance")]
session.stats[placement].describe().loc[["mean", "50%", "max"]].round(2)

In [ ]:
visualization.gap_figure(session, slice(0, 1200))

The distance lines **break wherever a side is contiguous**. That is not missing data: there
is no gap there, so there is no distance, and a step line drawn across it would assert one.

The statistics are cheap once the levels are in hand, so what a timing of them measures is
`occupied_levels` — which the dict rungs answer with `heapq.nsmallest` over the whole side,
and the tick array by walking exactly `reported_depth` bits.

In [ ]:
def gap_pass(book):
    for direction in (BUY, SELL):
        book.side_statistics(direction, DEPTH)

def fold_gaps(book_cls, opening, messages):
    book = seeded(book_cls, opening, messages)
    for message in messages:
        book.apply(message, record=False)
        gap_pass(book)

rows = []
for name, (opening, messages, final) in REGIMES.items():
    occupied = (len(final.levels_map(BUY)) + len(final.levels_map(SELL))) // 2
    timings = {"occupied levels a side": occupied}
    for book_cls in AXIS_B_VARIANTS:
        timings[book_cls.__name__] = round(
            best_of(lambda c=book_cls: fold_gaps(c, opening, messages)), 3
        )
    rows.append(pd.Series(timings, name=name))

pd.DataFrame(rows)

## 6. The ask side, and the one thing a band buys

`bit_length()` finds the **highest** set bit in constant time. The lowest one costs a scan:
`(bits & -bits)` builds a whole new integer. So on a single occupancy bitmap the bid side is
free and the ask side pays for the span at every lookup — which is why `TickArrayBook` used
to be the *slowest* rung at `occupied_levels(depth=10)`.

The fix is to stop asking for the lowest bit. `TickArrayBook` has a band, so it has a
ceiling, and the sell side is now indexed **downward from it**: bit `ceiling - price` rather
than `price - origin`. The best price on either side is then the highest set bit.

`BitmapBook` cannot do this. Its docstring is explicit that it has no upper edge — it grows
as prices arrive — and you cannot count down from an edge you do not have. So the step
between the two rungs now buys one more thing: *having* a band is what lets you reverse it.

In [ ]:
wide = TickArrayBook(origin=0, width=40_000)
for price in range(20_000, 22_000):
    wide.set_size(SELL, price, 10)
    wide.set_size(BUY, price - 10_000, 10)

def repeatedly(call, repeat=20_000):
    for _ in range(repeat):
        call()

print(f"best bid (highest set bit)   {best_of(lambda: repeatedly(lambda: wide.best_price(BUY))):.3f}s")
print(f"best ask (also highest, now) {best_of(lambda: repeatedly(lambda: wide.best_price(SELL))):.3f}s")
print(f"\nceiling = origin + width - 1 = {wide.ceiling}")

The two sides now cost the same, which is the whole point. The band's *upper* edge has become
load-bearing in exchange, so a book that shifted its band would have to move the ask bitmap
rather than merely extend the array — recorded in the class docstring, since it is the kind
of thing that is discovered painfully otherwise.

## 7. Three ideas that were measured and rejected

A measured rejection is worth as much as an accepted change, and rather harder to come by.

### 7a. Filling the LOBSTER padding with a strided slice

A row of a LOBSTER book is `4 x depth` integers; where a side has fewer occupied levels than
the depth, the rest is padding. The obvious improvement over a `while` loop writing two cells
at a time is one strided numpy store per side. It is slower.

In [ ]:
DEPTH_10 = 4 * 10

def by_scalar_writes(target, levels, column, padding):
    level = 0
    for price, resting in levels:
        target[4 * level + column] = price * PRICE_UNIT
        target[4 * level + column + 1] = resting
        level += 1
    while level < 10:
        target[4 * level + column] = padding
        target[4 * level + column + 1] = 0
        level += 1

def by_strided_fill(target, levels, column, padding):
    level = 0
    for price, resting in levels:
        target[4 * level + column] = price * PRICE_UNIT
        target[4 * level + column + 1] = resting
        level += 1
    if level < 10:
        target[4 * level + column::4] = padding
        target[4 * level + column + 1::4] = 0

rows = []
for occupied in (10, 6, 2):
    levels = [(10_000 + i, 100 + i) for i in range(occupied)]
    out = np.zeros((20_000, DEPTH_10), np.int64)
    timings = {"occupied levels": occupied}
    for label, writer in (("scalar writes", by_scalar_writes), ("strided fill", by_strided_fill)):
        timings[label] = round(best_of(
            lambda w=writer: [w(out[r], levels, 0, frames.ASK_PADDING) for r in range(20_000)]
        ), 3)
    rows.append(pd.Series(timings, name=f"{occupied} of 10"))
pd.DataFrame(rows)

A strided numpy store costs roughly a microsecond of fixed overhead, which is what four or
five scalar writes cost. The crossover is around eight padded cells a side, and at full depth
there are none.

**What does work is not writing the padding at all.** The buffer is filled with the padding
pattern once when it is allocated, and a row then writes only the levels the book has. That
is `frames.lobster_padding_row`, and the saving is largest exactly where the strided fill was
also best — a shallow book in a deep file.

There is a trap in it, and it is the reason the pre-fill lives inside `_RowBuffer` rather
than at the call site: the buffer doubles when it is full, growth allocates with `np.empty`,
and a grown tail that is not repainted hands freed memory to the frame as plausible small
integers that pass the schema.

In [ ]:
from unito26.lob.session import _book_buffer
from unito26.lob.statistics import SessionStatistics

buffer = _book_buffer(ReportedDepth(3), 0)          # 0: no length hint, so it will double
padding = frames.lobster_padding_row(ReportedDepth(3))
for _ in range(len(buffer.array) + 1):
    buffer.claim()
print(f"rows claimed: {buffer.used}, capacity now {len(buffer.array)}")
print(f"the row just past the growth boundary is still padding: "
      f"{bool((buffer.array[buffer.used - 1] == padding).all())}")

### 7b. Preallocating the buffer from the message count

A list of messages knows its own length, so the buffer can be sized exactly and never grow.
A generator does not — the simulator reads the book as it folds, so the stream cannot be
materialised first. `operator.length_hint` is exactly that distinction.

It is worth a couple of per cent, and is kept for something else.

In [ ]:
from operator import length_hint

opening, messages, _ = DEEP
print(f"length_hint(list)       {length_hint(messages, 0)}")
print(f"length_hint(iter(list)) {length_hint(iter(messages), 0)}   <- still exact")
print(f"length_hint(generator)  {length_hint((m for m in messages), 0)}")

sized = best_of(lambda: MarketSession.from_occupied_levels(
    seeded(TickArrayBook, opening, messages), messages, DEPTH, SPEC, PRICE_UNIT, False))
grown = best_of(lambda: MarketSession.from_occupied_levels(
    seeded(TickArrayBook, opening, messages), (m for m in messages),
    DEPTH, SPEC, PRICE_UNIT, False))
print(f"\nsized from the list  {sized:.4f}s")
print(f"doubled from nothing {grown:.4f}s")
print(f"saving               {100 * (grown - sized) / grown:.1f}%")

The reasons to keep it are not speed. The loop index becomes `enumerate`, so
`claim()` leaves the hot path — and with it the trap that `claim()`'s own docstring warns
about, where a reference to `.array` taken before the call is written into and then discarded.
And for a list of messages the growth path never runs at all, which is the path 7a can
corrupt.

Note the middle line above: `iter(list)` still reports an exact length. A test that wanted to
exercise the growth path and reached for `iter()` would silently not.

### 7c. Occupancy as a hierarchy of words

`BitmapBook`'s docstring says what C++ does: "a hierarchy of 64-bit words and a
count-trailing-zeros instruction". It then does the other thing, and holds the whole band in
one arbitrary-precision integer. Building the hierarchy — a list of words plus a summary
integer whose bit $w$ is set iff word $w$ is non-zero — is the obvious next rung.

It is not worth a class, and the reason is specific to Python.

In [ ]:
WORD, SHIFT, MASK = 32, 5, 31


class ChunkedTickArrayBook(TickArrayBook):
    """`TickArrayBook` with the occupancy in words instead of one arbitrary-precision int.

    A list of 32-bit words per side, plus a summary integer whose bit ``w`` is set iff word
    ``w`` is non-zero.  Everything else is inherited, so the comparison varies one thing:
    the mirrored ask side, the gap statistics read off the levels list and the accumulate
    profile are identical on both sides of it.
    """

    def __init__(self, origin, width, strict=False):
        super().__init__(origin, width, strict)
        size = (width >> SHIFT) + 1
        self._words = {BUY: [0] * size, SELL: [0] * size}
        self._summary = {BUY: 0, SELL: 0}

    def set_size(self, direction, price, size):
        slot = price - self.origin
        if not 0 <= slot < self.width:
            raise ValueError(f"price {price} is outside the band")
        self._sizes[direction][slot] = size
        index = slot if direction == BUY else self.ceiling - price
        at, bit = index >> SHIFT, index & MASK
        table = self._words[direction]
        table[at] = table[at] | (1 << bit) if size > 0 else table[at] & ~(1 << bit)
        if table[at]:
            self._summary[direction] |= 1 << at
        else:
            self._summary[direction] &= ~(1 << at)

    def best_price(self, direction):
        summary = self._summary[direction]
        if not summary:
            return None
        at = summary.bit_length() - 1
        index = (at << SHIFT) + self._words[direction][at].bit_length() - 1
        return self.origin + index if direction == BUY else self.ceiling - index

    def occupied_levels(self, direction, reported_depth):
        summary, table = self._summary[direction], self._words[direction]
        sizes, origin, ceiling = self._sizes[direction], self.origin, self.ceiling
        buying = direction == BUY
        found = []
        while summary and len(found) < reported_depth:
            at = summary.bit_length() - 1
            word, base = table[at], at << SHIFT
            while word and len(found) < reported_depth:
                index = base + word.bit_length() - 1
                price = origin + index if buying else ceiling - index
                found.append((price, sizes[price - origin]))
                word ^= 1 << (index - base)
            summary ^= 1 << at
        return found

    def levels_map(self, direction):
        return dict(self.occupied_levels(direction, self.width))

In [ ]:
def seeded_at_band(book_cls, width, opening, messages):
    prices = [p for p in (m.price for m in messages) if not is_market_price(p)]
    low, high = min(prices), max(prices)
    slack = max(0, width - (high - low)) // 2
    book = book_cls(origin=low - slack - 64, width=(high - low) + 2 * slack + 128)
    for direction in (BUY, SELL):
        for price, resting in opening.levels_map(direction).items():
            book.set_size(direction, price, resting)
    return book


# They must answer identically before it is worth timing them.
opening, messages, _ = DEEP
plain = seeded_at_band(TickArrayBook, 0, opening, messages)
chunked = seeded_at_band(ChunkedTickArrayBook, 0, opening, messages)
agree = True
for message in messages:
    plain.apply(message, record=False)
    chunked.apply(message, record=False)
    for direction in (BUY, SELL):
        agree &= plain.side_statistics(direction, DEPTH) == chunked.side_statistics(direction, DEPTH)
print("the two structures agree on every message:", agree)


def fold_with_statistics(book_cls, width, opening, messages):
    book = seeded_at_band(book_cls, width, opening, messages)
    for message in messages:
        book.apply(message, record=False)
        book.side_statistics(BUY, DEPTH)
        book.side_statistics(SELL, DEPTH)
        book.queue_imbalance_profile(LEVELS)


widths, one_integer, words_of_32 = [], [], []
for width in (0, 1_000, 4_000, 16_000, 64_000):
    widths.append(seeded_at_band(TickArrayBook, width, opening, messages).width)
    one_integer.append(best_of(
        lambda w=width: fold_with_statistics(TickArrayBook, w, opening, messages)))
    words_of_32.append(best_of(
        lambda w=width: fold_with_statistics(ChunkedTickArrayBook, w, opening, messages)))

timings = pd.DataFrame(
    {"one integer": one_integer, "words of 32": words_of_32},
    index=pd.Index(widths, name="band width (ticks)"),
)
timings["ratio"] = (timings["one integer"] / timings["words of 32"]).round(2)
timings.round(3)

In [ ]:
visualization.band_width_figure(
    timings[["one integer", "words of 32"]],
    "Occupancy as one integer, and as a hierarchy of words",
)

The lines cross, and they cross well above the band widths this material uses.

**Why it is unambiguously right in C++ and only conditionally right here.** In C++ a bitset
*is* a `uint64_t[]`; there is no arbitrary-precision integer to compare it against, so the
hierarchy is not an optimisation but the implementation, and the summary word skips empty
words with pure ALU work.

Python changes the trade twice over.

* `int.bit_length()` is already $O(1)$, so the hierarchy buys **nothing on the read**.
* What it could buy is on the **write**: Python integers are immutable, so `bits |= 1 << i`
  allocates and copies $\lceil \text{width}/30 \rceil$ digits. That cost grows with the band;
  the chunked write does not.

Against that, chunking *adds* a shift, a mask, a list index and an inner loop to every level
of every walk — a constant paid on every message. Below a few thousand ticks the added
interpreter overhead exceeds the digit copies it saves.

So the structure that is unconditionally correct in one language is a **crossover** in
another, and knowing where the crossover sits is the whole of the decision.

## 8. Where it ends up

Every rung, both regimes, the full fold with online statistics.

In [ ]:
rows = []
for name, (opening, messages, final) in REGIMES.items():
    timings = {}
    for book_cls in AXIS_B_VARIANTS:
        timings[book_cls.__name__] = round(best_of(
            lambda c=book_cls: MarketSession.from_occupied_levels(
                seeded(c, opening, messages), messages, DEPTH, SPEC, PRICE_UNIT, True)
        ), 3)
    rows.append(pd.Series(timings, name=name))

table = pd.DataFrame(rows)
table["fastest"] = table.idxmin(axis=1)
table

`TickArrayBook` is now the fastest rung in both regimes. Measured against the same code
before these changes, on the same machine and the same streams:

| full fold, online statistics | before | after |
| --- | --- | --- |
| shallow, `AggregateBook` | 0.318 | 0.193 |
| shallow, `TickArrayBook` | **0.365** | **0.165** |
| deep, `AggregateBook` | 0.905 | 0.401 |
| deep, `TickArrayBook` | 0.415 | 0.185 |

The bold row is the finding this notebook started from: the top of the ladder used to be
slower than the bottom of it. It is now 2.2 times faster than it was, and the shallow regime
— the one where an index has least to win — is where it took the longest to show.

None of the five changes was a better algorithm for finding a best price. They were: stop
sizing a band from a sentinel; stop asking the book the same question four times; stop
asking for the bit that is expensive to find; stop computing what the walk already computed;
and stop writing the padding you are about to overwrite. The ladder was never wrong. It was
being read in a place where it had nothing to say.